# 🏆 The Ultimate 1-Day Master: 5-Fold Cross-Validation with EfficientNetV2-L (750 Epochs)

สมุดงานนี้คือ **สูตรการเทรนระดับสูงสุด (Master-Grade Architecture)** ออกแบบมาให้รันคำนวณอย่างลึกซึ้งต่อเนื่องยาวนาน (~15 - 20 ชั่วโมง บน T4 GPU):

### 🌟 จุดเด่นของสถาปัตยกรรมชุดนี้:
1. **Backbone ระดับเรือธงสูงสุดของ Google:** `EfficientNetV2-L` (**118.5 ล้านพารามิเตอร์**)
2. **Ultra-High Resolution 384x384:** คมชัดระดับมองเห็นรอยยับเหล็ก ชิ้นส่วนแตกหัก ควันไฟ และคราบน้ำมันบนถนน
3. **5-Fold Cross-Validation Marathon (เทรน 5 สมองแยกกัน):**
   - โมเดล 5 ตัวจะสลับกันเรียนรู้คนละมุมมอง ตัวละ 150 Epochs (**รวมการคำนวณทั้งหมด 750 Epochs!**)
   - มี Test Set ส่วนกลาง (10%) ที่ไม่เคยผ่านตาโมเดลใดๆ มาก่อนเพื่อใช้วัดคะแนนเปรียบเทียบอย่างยุติธรรม
4. **คัดเลือก The Champion Model ลงมือถือ:**
   - เมื่อเทรนจบทั้ง 5 สมอง ระบบจะจัดอันดับ Leaderboard และดึง **โมเดลตัวที่ทำคะแนนสูงสุดอันดับ 1** ออกมาแปลงเป็น TFLite
5. **TFLite Heavyweight Size:** ~250 - 320 MB (พลังคำนวณเต็มพิกัด ตรงตามโควต้า 100-300 MB)
6. **Plug & Play กับ Flutter 100%:** ใช้งานกับ `accident_image_classifier_service.dart` ได้ทันที


In [ ]:
# ==== 1. เตรียมสภาพแวดล้อม GPU และไลบรารี ====
!pip install -q tensorflow scikit-learn matplotlib pillow

import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("✅ ใช้งาน GPU สำเร็จ:", gpus[0].name)
else:
    print("⚠️ ไม่พบ GPU — แนะนำไปที่เมนู: Runtime > Change runtime type > เลือก T4 GPU")

import numpy as np
import random, os, gc
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)


In [ ]:
# ==== 2. นำเข้า dataset.zip (191 MB ตัวคลีน) ====
import os, zipfile, shutil, pathlib

shutil.rmtree("dataset", ignore_errors=True)
shutil.rmtree("dataset_folds", ignore_errors=True)
shutil.rmtree("_extract", ignore_errors=True)

USE_GOOGLE_DRIVE = False  # เปลี่ยนเป็น True หากวาง dataset.zip ไว้ใน Google Drive

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    zip_path = "/content/drive/MyDrive/dataset.zip"
    print(f"กำลังแตกไฟล์จาก Google Drive: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("_extract")
else:
    from google.colab import files
    print("กรุณาเลือกไฟล์ dataset.zip (191 MB) จากหน้า Desktop ของคุณ:")
    uploaded = files.upload()
    zip_file = next(iter(uploaded))
    with zipfile.ZipFile(zip_file, "r") as zip_ref:
        zip_ref.extractall("_extract")

for junk in pathlib.Path("_extract").rglob("__MACOSX"):
    shutil.rmtree(junk, ignore_errors=True)

found = [d for d in pathlib.Path("_extract").rglob("accident") if d.is_dir() and (d.parent / "non_accident").is_dir()]
assert found, "❌ ไม่พบโฟลเดอร์ accident/ กับ non_accident/ ในไฟล์ zip"

pathlib.Path("dataset").mkdir(exist_ok=True)
for name in ["accident", "non_accident"]:
    target = found[0].parent / name
    dest = pathlib.Path("dataset") / name
    if dest.exists():
        shutil.rmtree(dest)
    shutil.move(str(target), str(dest))
shutil.rmtree("_extract", ignore_errors=True)

print("✅ แตกไฟล์เรียบร้อย โฟลเดอร์ที่พร้อมใช้งาน:", sorted(p.name for p in pathlib.Path("dataset").iterdir() if p.is_dir()))


In [ ]:
# ==== 3. ตรวจสอบไฟล์ภาพและสรุปยอดข้อมูล ====
import pathlib, os
from PIL import Image

def scrub_invalid_images(root="dataset"):
    removed = 0
    total = 0
    for class_dir in pathlib.Path(root).iterdir():
        if not class_dir.is_dir(): continue
        for img_path in list(class_dir.glob("*")):
            if img_path.name.startswith("."): 
                img_path.unlink(); continue
            total += 1
            try:
                with Image.open(img_path) as im: im.verify()
            except Exception:
                try: os.remove(img_path); removed += 1
                except Exception: pass
    print(f"ตรวจสอบเสร็จสิ้น: ทั้งหมด {total:,} รูป (ลบไฟล์เสีย {removed} ไฟล์)")

scrub_invalid_images()

data_dir = pathlib.Path("dataset")
class_dirs = sorted([d for d in data_dir.iterdir() if d.is_dir()])
for d in class_dirs:
    count = len([f for f in d.glob("*") if not f.name.startswith(".")])
    print(f"📁 {d.name}: {count:,} รูป")


In [ ]:
# ==== 4. สร้าง 5-Fold Stratified Cross-Validation + Hold-out Test Set (384x384) ====
import pathlib, shutil
from sklearn.model_selection import StratifiedKFold, train_test_split
import tensorflow as tf

IMG_SIZE = 384  # ความละเอียดระดับสูงพิเศษ 384x384 สำหรับ EfficientNetV2-L
BATCH_SIZE = 16 # เหมาะสำหรับโมเดล 118M params และภาพ 384 บน T4 GPU VRAM 16GB

folds_root = pathlib.Path("dataset_folds")
shutil.rmtree(folds_root, ignore_errors=True)

classes = ["accident", "non_accident"]
all_files = []
all_labels = []
for c in classes:
    c_files = [f for f in (pathlib.Path("dataset") / c).glob("*") if f.is_file() and not f.name.startswith(".")]
    for f in c_files:
        all_files.append(f)
        all_labels.append(c)

# แยก Hold-out Test Set 10% ไว้ทดสอบร่วมกันตอนท้ายสำหรับโมเดลทั้ง 5 ตัว
train_val_files, test_files, train_val_labels, test_labels = train_test_split(
    all_files, all_labels, test_size=0.10, random_state=42, stratify=all_labels
)

# บันทึก Hold-out Test Set
for c in classes:
    (folds_root / "test" / c).mkdir(parents=True, exist_ok=True)
for f, l in zip(test_files, test_labels):
    shutil.copy(f, folds_root / "test" / l / f.name)

print(f"🎯 Hold-out Test Set (Unseen): {len(test_files)} รูป")

# ทำ 5-Fold Stratified K-Fold บนข้อมูล 90% ที่เหลือ
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_val_files, train_val_labels), 1):
    for c in classes:
        (folds_root / f"fold_{fold_idx}" / "train" / c).mkdir(parents=True, exist_ok=True)
        (folds_root / f"fold_{fold_idx}" / "val" / c).mkdir(parents=True, exist_ok=True)
    
    for idx in train_idx:
        f, l = train_val_files[idx], train_val_labels[idx]
        shutil.copy(f, folds_root / f"fold_{fold_idx}" / "train" / l / f.name)
    for idx in val_idx:
        f, l = train_val_files[idx], train_val_labels[idx]
        shutil.copy(f, folds_root / f"fold_{fold_idx}" / "val" / l / f.name)
    
    print(f"  [Fold {fold_idx}] Train: {len(train_idx):,} รูป | Val: {len(val_idx):,} รูป")

# Data Augmentation จำลองทุกสภาพแวดล้อมกล้องมือถือ/หน้ารถจริง
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.20),
    tf.keras.layers.RandomZoom(0.25),
    tf.keras.layers.RandomContrast(0.25),
    tf.keras.layers.RandomTranslation(0.15, 0.15),
    tf.keras.layers.RandomBrightness(0.25),
    tf.keras.layers.GaussianNoise(0.05),
], name="master_data_augmentation")

# เตรียม Test Dataset ส่วนกลาง
AUTOTUNE = tf.data.AUTOTUNE
test_ds = tf.keras.utils.image_dataset_from_directory(
    folds_root / "test",
    label_mode="categorical",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
).prefetch(buffer_size=AUTOTUNE)

class_names = ["accident", "non_accident"]
print("✅ เตรียม 5 Folds และ Test Set 384x384 เรียบร้อย!")


In [ ]:
# ==== 5. ฟังก์ชันสร้างโมเดลยักษ์ EfficientNetV2-L (118.5 ล้านพารามิเตอร์) ====
preprocess_input = tf.keras.applications.efficientnet_v2.preprocess_input

def build_titan_model():
    base_model = tf.keras.applications.EfficientNetV2L(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet",
    )
    base_model.trainable = False
    
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_layer")
    x = data_augmentation(inputs)
    x = preprocess_input(x)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="avg_pool")(x)
    x = tf.keras.layers.BatchNormalization(name="batch_norm")(x)
    x = tf.keras.layers.Dense(256, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(1e-4), name="dense_256")(x)
    x = tf.keras.layers.Dropout(0.4, name="dropout_1")(x)
    x = tf.keras.layers.Dense(64, activation="relu", name="dense_64")(x)
    x = tf.keras.layers.Dropout(0.2, name="dropout_2")(x)
    outputs = tf.keras.layers.Dense(len(class_names), activation="softmax", name="prediction")(x)
    
    model = tf.keras.Model(inputs, outputs, name="titan_efficientnetv2l")
    return model, base_model

sample_model, _ = build_titan_model()
print(f"👑 สถาปัตยกรรม The Titan: {sample_model.count_params():,} พารามิเตอร์ พร้อมรันมาราธอน 5 Folds!")
del sample_model
tf.keras.backend.clear_session()
gc.collect()


In [ ]:
# ==== 6. เริ่มการเทรนมาราธอน 5-Fold Cross-Validation (750 Epochs / 15-20 ชม.) ====
import time

N_FOLDS = 5
fold_histories = {}
fold_test_scores = {}

loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

marathon_start = time.time()
print("🏁 เริ่มต้นการเทรนระดับ Master 5 Folds x 150 Epochs = 750 Epochs รวม!")

for fold_num in range(1, N_FOLDS + 1):
    fold_start = time.time()
    print(f"\n{'='*70}")
    print(f"🔥 กำลังเทรนสมองตัวที่ [{fold_num}/{N_FOLDS}] - FOLD {fold_num}")
    print(f"{'='*70}")
    
    train_dir = folds_root / f"fold_{fold_num}" / "train"
    val_dir = folds_root / f"fold_{fold_num}" / "val"
    
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, label_mode="categorical", image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, shuffle=True, seed=42
    ).shuffle(500).prefetch(buffer_size=AUTOTUNE)
    
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir, label_mode="categorical", image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, shuffle=False
    ).prefetch(buffer_size=AUTOTUNE)
    
    # คำนวณ class weight
    n_acc = len(list((train_dir / "accident").glob("*")))
    n_non = len(list((train_dir / "non_accident").glob("*")))
    tot = n_acc + n_non
    c_weight = {0: tot / (2 * n_acc), 1: tot / (2 * n_non)}
    
    # สร้างโมเดลใหม่สำหรับ Fold นี้
    model, base_model = build_titan_model()
    
    checkpoint_path = f"best_model_fold_{fold_num}.keras"
    ckpt_cb = tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path, monitor="val_accuracy", save_best_only=True, mode="max", verbose=1
    )
    
    steps_per_epoch = len(train_ds)
    
    # --- Phase 1: Warmup Head (30 Epochs) ---
    EPOCHS_P1 = 30
    lr_p1 = tf.keras.optimizers.schedules.CosineDecay(1e-3, EPOCHS_P1 * steps_per_epoch, alpha=0.01)
    try: opt_p1 = tf.keras.optimizers.AdamW(learning_rate=lr_p1, weight_decay=1e-4)
    except: opt_p1 = tf.keras.optimizers.Adam(learning_rate=lr_p1)
    model.compile(optimizer=opt_p1, loss=loss_fn, metrics=["accuracy"])
    
    print(f"  [Fold {fold_num}] 🚀 Phase 1: Warmup Head 30 Epochs...")
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_P1, class_weight=c_weight,
                   callbacks=[ckpt_cb, tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=0)])
    
    # --- Phase 2: Mid-Level Fine-Tuning 50% (60 Epochs) ---
    base_model.trainable = True
    mid_at = int(len(base_model.layers) * 0.5)
    for l in base_model.layers[:mid_at]: l.trainable = False
    for l in base_model.layers[mid_at:]:
        if isinstance(l, tf.keras.layers.BatchNormalization): l.trainable = False
    
    EPOCHS_P2 = 60
    lr_p2 = tf.keras.optimizers.schedules.CosineDecay(5e-5, EPOCHS_P2 * steps_per_epoch, alpha=0.01)
    try: opt_p2 = tf.keras.optimizers.AdamW(learning_rate=lr_p2, weight_decay=1e-4)
    except: opt_p2 = tf.keras.optimizers.Adam(learning_rate=lr_p2)
    model.compile(optimizer=opt_p2, loss=loss_fn, metrics=["accuracy"])
    
    ep_init_p2 = len(h1.history["accuracy"])
    print(f"  [Fold {fold_num}] 🚀 Phase 2: Fine-Tuning 50% อีก 60 Epochs...")
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=ep_init_p2 + EPOCHS_P2, initial_epoch=ep_init_p2,
                   class_weight=c_weight, callbacks=[ckpt_cb, tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=12, restore_best_weights=True, verbose=0)])
    
    # --- Phase 3: Ultra-Deep Fine-Tuning 85% (60 Epochs) ---
    deep_at = int(len(base_model.layers) * 0.15)
    for l in base_model.layers[:deep_at]: l.trainable = False
    for l in base_model.layers[deep_at:]:
        if isinstance(l, tf.keras.layers.BatchNormalization): l.trainable = False
    
    EPOCHS_P3 = 60
    lr_p3 = tf.keras.optimizers.schedules.CosineDecay(5e-6, EPOCHS_P3 * steps_per_epoch, alpha=0.05)
    try: opt_p3 = tf.keras.optimizers.AdamW(learning_rate=lr_p3, weight_decay=1e-4)
    except: opt_p3 = tf.keras.optimizers.Adam(learning_rate=lr_p3)
    model.compile(optimizer=opt_p3, loss=loss_fn, metrics=["accuracy"])
    
    ep_init_p3 = len(h1.history["accuracy"]) + len(h2.history["accuracy"])
    print(f"  [Fold {fold_num}] 🚀 Phase 3: Ultra-Deep Fine-Tuning 85% อีก 60 Epochs...")
    h3 = model.fit(train_ds, validation_data=val_ds, epochs=ep_init_p3 + EPOCHS_P3, initial_epoch=ep_init_p3,
                   class_weight=c_weight, callbacks=[ckpt_cb, tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True, verbose=0)])
    
    # โหลด Best Weight ของ Fold นี้มาวัดผลบน Test Set กลาง
    if os.path.exists(checkpoint_path):
        best_fold_model = tf.keras.models.load_model(checkpoint_path)
    else:
        best_fold_model = model
        
    t_loss, t_acc = best_fold_model.evaluate(test_ds, verbose=0)
    fold_test_scores[fold_num] = t_acc
    fold_elapsed = (time.time() - fold_start) / 3600
    print(f"  🎯 Fold {fold_num} สำเร็จ! Test Accuracy: {t_acc*100:.2f}% (ใช้เวลา {fold_elapsed:.2f} ชม.)")
    
    # เคลียร์ Memory ป้องกัน RAM ค้าง
    del model, base_model, best_fold_model
    tf.keras.backend.clear_session()
    gc.collect()

total_marathon_hours = (time.time() - marathon_start) / 3600
print(f"\n🏆 จบการเทรนมาราธอน 5 Folds สมบูรณ์แบบ! รวมเวลาทั้งหมด: {total_marathon_hours:.2f} ชั่วโมง")


In [ ]:
# ==== 7. สรุปตารางคะแนน Leaderboard และคัดเลือก The Champion Model ====
print("="*60)
print("👑 5-FOLD MARATHON LEADERBOARD (คะแนนบน Unseen Test Set)")
print("="*60)
for f_num, score in sorted(fold_test_scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  Fold {f_num}: Accuracy = {score*100:.2f}%")

# ค้นหาแชมป์อันดับ 1
champion_fold = max(fold_test_scores, key=fold_test_scores.get)
champion_acc = fold_test_scores[champion_fold]
champion_model_path = f"best_model_fold_{champion_fold}.keras"

print("="*60)
print(f"🥇 THE CHAMPION MODEL: FOLD {champion_fold} (ความแม่นยำสูงสุด: {champion_acc*100:.2f}%)")
print(f"📦 กำลังโหลดโมเดลแชมป์: {champion_model_path}")
print("="*60)

champion_model = tf.keras.models.load_model(champion_model_path)


In [ ]:
# ==== 8. ประเมินผลเต็มรูปแบบของ The Champion Model บน Test Set ====
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = champion_model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("📊 Classification Report ของโมเดลแชมป์ (Champion Model):")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, fontsize=11); ax.set_yticklabels(class_names, fontsize=11)
ax.set_xlabel("ทำนาย (Predicted)", fontsize=12); ax.set_ylabel("จริง (Actual)", fontsize=12)

for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=12, fontweight="bold")
plt.colorbar(im)
plt.title(f"Confusion Matrix (Champion Fold {champion_fold} - {len(y_true):,} รูป)", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ==== 9. ดูตัวอย่างภาพที่โมเดลแชมป์ทำนายผิด (ถ้ามี) ====
mis_images, mis_true, mis_pred = [], [], []
for images, labels in test_ds:
    preds = champion_model.predict(images, verbose=0)
    p_labels = np.argmax(preds, axis=1)
    t_labels = np.argmax(labels.numpy(), axis=1)
    for i in range(len(labels)):
        if p_labels[i] != t_labels[i]:
            mis_images.append(images[i].numpy().astype("uint8"))
            mis_true.append(t_labels[i])
            mis_pred.append(p_labels[i])

print(f"พบภาพที่ทำนายผิด {len(mis_images)} จาก {len(y_true)} รูปใน Test Set")
n_show = min(9, len(mis_images))
if n_show > 0:
    plt.figure(figsize=(10, 10))
    for i in range(n_show):
        plt.subplot(3, 3, i + 1)
        plt.imshow(mis_images[i])
        plt.title(f"จริง: {class_names[mis_true[i]]}\nทำนาย: {class_names[mis_pred[i]]}", fontsize=10, color="red")
        plt.axis("off")
    plt.suptitle("ภาพที่โมเดลทำนายผิดใน Test Set", fontsize=13)
    plt.tight_layout(); plt.show()


In [ ]:
# ==== 10. แปลง The Champion Model เป็น TFLite Heavyweight (~250 - 320 MB) ====
converter = tf.lite.TFLiteConverter.from_keras_model(champion_model)
tflite_model = converter.convert()

tflite_filename = "accident_classifier.tflite"
labels_filename = "accident_labels.txt"

with open(tflite_filename, "wb") as f:
    f.write(tflite_model)

with open(labels_filename, "w") as f:
    for name in class_names:
        f.write(name + "\n")

size_mb = len(tflite_model) / (1024 * 1024)
print(f"🏆 บันทึกโมเดลแชมป์สำเร็จ: {tflite_filename}")
print(f"📦 ขนาดไฟล์ TFLite: {size_mb:.2f} MB (พารามิเตอร์ 118 ล้านตัว พลังเต็มร้อย!)")
print(f"📝 บันทึก Label สำเร็จ: {labels_filename}")


In [ ]:
# ==== 11. ตรวจสอบความถูกต้องของ TFLite เทียบกับโมเดลแชมป์ต้นฉบับ ====
interpreter = tf.lite.Interpreter(model_path="accident_classifier.tflite")
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite Input:", input_details[0]["shape"], input_details[0]["dtype"])
print("TFLite Output:", output_details[0]["shape"], output_details[0]["dtype"])

sample_images, sample_labels = next(iter(test_ds.take(1)))
match_count = 0
n_samples = len(sample_images)

for i in range(n_samples):
    img_input = tf.expand_dims(sample_images[i], 0)
    k_pred = np.argmax(champion_model.predict(img_input, verbose=0)[0])
    
    interpreter.set_tensor(input_details[0]["index"], img_input.numpy())
    interpreter.invoke()
    tf_pred = np.argmax(interpreter.get_tensor(output_details[0]["index"])[0])
    
    if k_pred == tf_pred: match_count += 1

print(f"🎯 ผลลัพธ์ตรงกัน {match_count}/{n_samples} รูป ระหว่าง Keras กับ TFLite")
if match_count == n_samples:
    print("✅ การแปลงไฟล์ TFLite สมบูรณ์ 100% พร้อมนำไปใส่ในโปรเจกต์ Flutter")


In [ ]:
# ==== 12. ดาวน์โหลด The Champion Model สำหรับ Flutter App ====
from google.colab import files

files.download("accident_classifier.tflite")
files.download("accident_labels.txt")
print("🎉 ดาวน์โหลดโมเดลแชมป์เรียบร้อย พร้อมนำไปใส่ใน assets/models/ ของแอป Flutter!")
